# End-to-End Pipeline v3: PDF → KPI Extraction → Analytical Report → Market Decision

Версия v3 без OCR fallback, но с усилениями:

1. Более сильный `section_hint`
   - header-zone matching
   - fuzzy-like scoring по словарям
   - section type + section score
2. Словарный слой до LLM
   - прямой маппинг очевидных KPI
   - уменьшение числа LLM-вызовов
3. Нормализация масштаба `тыс./млн/млрд/трлн`
4. Автоматическое извлечение периода
5. Сохранение промежуточных кандидатов для отладки
6. Аналитический отчёт и investment decision

Архитектура:

```text
PDF
→ parse text + tables
→ strong section hinting
→ candidate generation
→ dictionary gate
→ LLM normalization only for hard cases
→ consolidation
→ derived metrics
→ analytical report
→ market decision
```


## 1. Установка зависимостей

In [ ]:
!pip -q install pdfplumber transformers accelerate torch pymupdf sentencepiece bitsandbytes

## 2. Загрузка PDF в Colab

In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_path = next(iter(uploaded.keys()))
print("PDF:", pdf_path)


## 3. Импорты

In [ ]:
import json
import re
import difflib
from dataclasses import dataclass, asdict
from typing import List, Optional, Dict, Any, Tuple

import pdfplumber
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig


## 4. Конфигурация и загрузка модели

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
USE_4BIT = True
DEVICE_MAP = "auto"

CANONICAL_KPIS = [
    "revenue",
    "gross_profit",
    "operating_income",
    "ebitda",
    "net_income",
    "eps",
    "total_assets",
    "equity",
    "cash",
    "debt",
    "operating_cash_flow",
    "capex",
]

SECTION_PATTERNS = {
    "financial_highlights": [
        "ключевые финансовые показатели",
        "основные финансовые показатели",
        "финансовые результаты",
        "financial highlights",
    ],
    "income_statement": [
        "отчет о прибыли",
        "отчет о прибылях и убытках",
        "консолидированный отчет о прибыли",
        "income statement",
        "statement of profit",
    ],
    "balance_sheet": [
        "отчет о финансовом положении",
        "бухгалтерский баланс",
        "statement of financial position",
        "balance sheet",
    ],
    "cash_flow": [
        "отчет о движении денежных средств",
        "cash flow statement",
        "statement of cash flows",
    ],
    "management_commentary": [
        "управленческий комментарий",
        "обзор результатов",
        "management discussion",
        "md&a",
    ],
}

BAD_PATTERNS = [
    "омб",
    "omb",
    "страница",
    "оглавление",
]

SOURCE_PRIORITY = {
    "table_row": 2,
    "text_pair": 1,
}

KPI_SYNONYMS = {
    "revenue": [
        "выручка", "доходы", "оборот", "revenue", "sales", "turnover"
    ],
    "gross_profit": [
        "валовая прибыль", "gross profit"
    ],
    "operating_income": [
        "операционная прибыль", "прибыль от операционной деятельности", "operating income", "operating profit"
    ],
    "ebitda": [
        "ebitda", "скорректированная ebitda", "adjusted ebitda"
    ],
    "net_income": [
        "чистая прибыль", "прибыль за период", "net income", "net profit", "profit attributable"
    ],
    "eps": [
        "eps", "прибыль на акцию", "earnings per share"
    ],
    "total_assets": [
        "активы", "итого активы", "total assets"
    ],
    "equity": [
        "капитал", "собственный капитал", "equity"
    ],
    "cash": [
        "денежные средства", "денежные средства и их эквиваленты", "cash", "cash and cash equivalents"
    ],
    "debt": [
        "долг", "чистый долг", "обязательства по кредитам и займам", "debt", "borrowings"
    ],
    "operating_cash_flow": [
        "денежный поток от операционной деятельности", "операционный денежный поток", "operating cash flow"
    ],
    "capex": [
        "капитальные затраты", "капитальные вложения", "capex"
    ],
}

bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map=DEVICE_MAP,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    quantization_config=bnb_config,
)
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("Loaded model:", MODEL_NAME)
print("4-bit quantization:", USE_4BIT)
print("CUDA available:", torch.cuda.is_available())


## 5. Структуры данных

In [ ]:
@dataclass
class Candidate:
    source_type: str
    page_num: int
    section_hint: Optional[str]
    section_type: Optional[str]
    section_score: float
    label_text: str
    value_text: str
    raw_text: str
    normalized_value_text: Optional[str] = None
    extracted_period: Optional[str] = None
    pre_mapped_kpi: Optional[str] = None
    pre_map_confidence: float = 0.0


@dataclass
class NormalizedKPI:
    canonical_kpi: Optional[str]
    value: Optional[float]
    unit: Optional[str]
    period: Optional[str]
    is_kpi: bool
    confidence: float
    reason: str
    source_type: str
    page_num: int
    section_hint: Optional[str]
    section_type: Optional[str]
    section_score: float
    label_text: str
    value_text: str
    normalized_value_text: Optional[str]
    extracted_period: Optional[str]
    raw_text: str
    normalization_source: str


## 6. Section hinting, период и нормализация масштаба

In [ ]:
def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()


def fuzzy_contains_score(text: str, phrase: str) -> float:
    text = text.lower()
    phrase = phrase.lower()
    if phrase in text:
        return 1.0
    # простая approximate similarity
    return difflib.SequenceMatcher(None, text[: min(len(text), 500)], phrase).ratio()


def detect_section_info(text: str) -> Dict[str, Any]:
    if not text:
        return {
            "section_hint": None,
            "section_type": None,
            "section_score": 0.0,
        }

    lines = [line.strip().lower() for line in text.split("\n") if line.strip()]
    header_zone = " ".join(lines[:25])

    best_type = None
    best_hint = None
    best_score = 0.0

    for section_type, phrases in SECTION_PATTERNS.items():
        for phrase in phrases:
            score = fuzzy_contains_score(header_zone, phrase)
            if phrase in header_zone:
                score += 0.5
            if score > best_score:
                best_score = score
                best_type = section_type
                best_hint = phrase

    if best_score < 0.55:
        return {
            "section_hint": None,
            "section_type": None,
            "section_score": 0.0,
        }

    return {
        "section_hint": best_hint,
        "section_type": best_type,
        "section_score": round(min(best_score, 1.5), 3),
    }


def extract_period_from_text(text: str) -> Optional[str]:
    if not text:
        return None

    t = text.lower()

    patterns = [
        r"(?:1|2|3|4)\s*кв\.?\s*20\d{2}",
        r"(?:i|ii|iii|iv)\s*кв\.?\s*20\d{2}",
        r"(?:6|9|12)\s*мес\.?\s*20\d{2}",
        r"за\s+\d+\s+месяцев\s+20\d{2}",
        r"fy\s*20\d{2}",
        r"20\d{2}\s*год",
        r"20\d{2}",
    ]

    for pat in patterns:
        m = re.search(pat, t, flags=re.IGNORECASE)
        if m:
            return m.group(0)
    return None


MULTIPLIERS = {
    "тыс": 1_000,
    "тыс.": 1_000,
    "thousand": 1_000,
    "млн": 1_000_000,
    "million": 1_000_000,
    "млрд": 1_000_000_000,
    "billion": 1_000_000_000,
    "трлн": 1_000_000_000_000,
    "trillion": 1_000_000_000_000,
}


def detect_unit(text: str) -> Optional[str]:
    if not text:
        return None
    t = text.lower()
    if "%" in t:
        return "PERCENT"
    if "руб" in t or "₽" in t:
        return "RUB"
    if "usd" in t or "$" in t or "долл" in t:
        return "USD"
    if "eur" in t or "€" in t:
        return "EUR"
    return None


def normalize_scale_in_value_text(value_text: str) -> Tuple[Optional[float], Optional[str], Optional[str]]:
    if not value_text:
        return None, None, None

    original = value_text
    t = value_text.lower().replace(" ", "")
    t = t.replace(",", ".")
    t = t.replace("−", "-").replace("–", "-")

    m = re.search(r"-?\d+(?:\.\d+)?", t)
    if not m:
        return None, detect_unit(original), None

    num = float(m.group(0))

    multiplier = 1
    for scale, mult in MULTIPLIERS.items():
        if scale in t:
            multiplier = mult
            break

    normalized_value = num * multiplier
    if abs(normalized_value - round(normalized_value)) < 1e-9:
        normalized_value = int(round(normalized_value))

    unit = detect_unit(original)
    normalized_text = str(normalized_value)

    return normalized_value, unit, normalized_text


def pre_map_label_to_kpi(label_text: str) -> Tuple[Optional[str], float]:
    if not label_text:
        return None, 0.0

    label = label_text.lower().strip()

    best_kpi = None
    best_score = 0.0

    for canonical_kpi, synonyms in KPI_SYNONYMS.items():
        for synonym in synonyms:
            synonym = synonym.lower()
            if synonym == label or synonym in label:
                score = 0.95 if synonym == label else 0.82
            else:
                score = difflib.SequenceMatcher(None, label, synonym).ratio()

            if score > best_score:
                best_score = score
                best_kpi = canonical_kpi

    if best_score < 0.72:
        return None, 0.0

    return best_kpi, round(float(best_score), 3)


## 7. PDF parsing и candidate generation

In [ ]:
def extract_pages(pdf_path: str) -> List[Dict[str, Any]]:
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            tables = page.extract_tables() or []
            section_info = detect_section_info(text)
            pages.append({
                "page_num": i,
                "text": text,
                "tables": tables,
                **section_info,
            })
    return pages


LABEL_VALUE_REGEXES = [
    re.compile(
        r"(?P<label>[А-Яа-яA-Za-zЁё0-9\-\s()/%,\.]{3,100})\s*[:\-–]\s*(?P<value>[+\-]?\d[\d\s,\.]*(?:млн|млрд|тыс\.|трлн)?\s*(?:руб\.|рублей|₽|%|долл\.|usd|eur)?)",
        re.IGNORECASE
    ),
    re.compile(
        r"(?P<label>Выручка|EBITDA|Чистая прибыль|Операционная прибыль|Прибыль за период|Денежные средства(?: и их эквиваленты)?|Капитальные затраты|Капитал|Собственный капитал|Активы|Обязательства|Долг)\s+(?:составила|составил|достигла|достиг|равна|равен|увеличилась до|снизилась до)\s+(?P<value>[+\-]?\d[\d\s,\.]*(?:млн|млрд|тыс\.|трлн)?\s*(?:руб\.|рублей|₽|%|долл\.|usd|eur)?)",
        re.IGNORECASE
    )
]


def looks_like_bad_candidate(label: str, value: str, raw_text: str) -> bool:
    blob = f"{label} {value} {raw_text}".lower()
    return any(p in blob for p in BAD_PATTERNS)


def make_candidate(page: Dict[str, Any], source_type: str, label: str, value: str, raw_text: str) -> Candidate:
    _, _, normalized_value_text = normalize_scale_in_value_text(value)
    extracted_period = extract_period_from_text(raw_text) or extract_period_from_text(page.get("text", "")) or extract_period_from_text(page.get("section_hint") or "")
    pre_mapped_kpi, pre_map_confidence = pre_map_label_to_kpi(label)

    return Candidate(
        source_type=source_type,
        page_num=page["page_num"],
        section_hint=page.get("section_hint"),
        section_type=page.get("section_type"),
        section_score=float(page.get("section_score", 0.0)),
        label_text=label,
        value_text=value,
        raw_text=raw_text,
        normalized_value_text=normalized_value_text,
        extracted_period=extracted_period,
        pre_mapped_kpi=pre_mapped_kpi,
        pre_map_confidence=pre_map_confidence,
    )


def generate_table_candidates(page: Dict[str, Any]) -> List[Candidate]:
    out = []

    for table in page["tables"]:
        if not table:
            continue

        for row in table:
            if not row:
                continue

            cells = [clean_text(c) for c in row if c and clean_text(c)]
            if len(cells) < 2:
                continue

            label = cells[0]
            values = cells[1:]

            value_candidate = None
            for v in values:
                if re.search(r"\d", v):
                    value_candidate = v
                    break

            if not value_candidate:
                continue

            raw = " | ".join(cells)

            if looks_like_bad_candidate(label, value_candidate, raw):
                continue

            if re.search(r"(выруч|ebitda|прибыл|актив|капитал|денежн|долг|обязат|eps)", label, re.IGNORECASE):
                out.append(make_candidate(page, "table_row", label, value_candidate, raw))

    return out


def generate_text_candidates(page: Dict[str, Any]) -> List[Candidate]:
    out = []
    lines = [clean_text(x) for x in page["text"].split("\n") if clean_text(x)]

    for line in lines:
        for rx in LABEL_VALUE_REGEXES:
            m = rx.search(line)
            if not m:
                continue

            label = clean_text(m.group("label"))
            value = clean_text(m.group("value"))

            if looks_like_bad_candidate(label, value, line):
                continue

            out.append(make_candidate(page, "text_pair", label, value, line))
            break

    return out


def generate_candidates(pages: List[Dict[str, Any]]) -> List[Candidate]:
    all_candidates = []
    for page in pages:
        all_candidates.extend(generate_table_candidates(page))
        all_candidates.extend(generate_text_candidates(page))
    return all_candidates


## 8. Dictionary gate + LLM normalization

In [ ]:
def extract_json(text: str) -> Optional[Dict[str, Any]]:
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


NORMALIZATION_PROMPT = """
You are a financial information extraction system.

Task:
Given one candidate extracted from a Russian corporate report, determine whether it is a valid KPI.
If yes, map it to one canonical KPI from the allowed list and normalize the numeric value.

Allowed canonical KPI list:
{canonical_kpis}

Rules:
1. Return valid JSON only.
2. If the candidate is not a KPI, set "is_kpi": false and "canonical_kpi": null.
3. Use candidate.normalized_value_text when it is reasonable.
4. If period is available in candidate.extracted_period, use it unless clearly wrong.
5. If unit is rubles, use "RUB". If percent, use "PERCENT". If unknown, null.
6. Ignore boilerplate, page numbers, regulatory metadata, and non-financial counters.
7. Prefer the simplest correct mapping.

Candidate:
{candidate_json}

Return JSON with this schema:
{{
  "canonical_kpi": "revenue" | "gross_profit" | "operating_income" | "ebitda" | "net_income" |
                   "eps" | "total_assets" | "equity" | "cash" | "debt" |
                   "operating_cash_flow" | "capex" | null,
  "value": number | null,
  "unit": "RUB" | "USD" | "EUR" | "PERCENT" | null,
  "period": string | null,
  "is_kpi": true | false,
  "confidence": number,
  "reason": string
}}
""".strip()


def run_llm_json(prompt: str, max_new_tokens: int = 350) -> Dict[str, Any]:
    messages = [
        {"role": "system", "content": "Return valid JSON only."},
        {"role": "user", "content": prompt},
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    output = generator(
        formatted_prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0
    )

    raw_text = output[0]["generated_text"][len(formatted_prompt):]
    parsed = extract_json(raw_text)

    return {
        "prompt": prompt,
        "raw_response": raw_text,
        "parsed": parsed
    }


def normalize_candidate_with_dictionary(candidate: Candidate) -> Optional[NormalizedKPI]:
    if not candidate.pre_mapped_kpi or candidate.pre_map_confidence < 0.88:
        return None

    normalized_value, unit, _ = normalize_scale_in_value_text(candidate.value_text)
    if normalized_value is None:
        return None

    return NormalizedKPI(
        canonical_kpi=candidate.pre_mapped_kpi,
        value=normalized_value,
        unit=unit,
        period=candidate.extracted_period,
        is_kpi=True,
        confidence=min(0.97, candidate.pre_map_confidence),
        reason=f"Dictionary-mapped from label '{candidate.label_text}'",
        source_type=candidate.source_type,
        page_num=candidate.page_num,
        section_hint=candidate.section_hint,
        section_type=candidate.section_type,
        section_score=candidate.section_score,
        label_text=candidate.label_text,
        value_text=candidate.value_text,
        normalized_value_text=candidate.normalized_value_text,
        extracted_period=candidate.extracted_period,
        raw_text=candidate.raw_text,
        normalization_source="dictionary"
    )


def llm_normalize_candidate(candidate: Candidate) -> Optional[NormalizedKPI]:
    prompt = NORMALIZATION_PROMPT.format(
        canonical_kpis=json.dumps(CANONICAL_KPIS, ensure_ascii=False),
        candidate_json=json.dumps(asdict(candidate), ensure_ascii=False, indent=2)
    )

    result = run_llm_json(prompt, max_new_tokens=250)
    data = result["parsed"]
    if not data:
        return None

    return NormalizedKPI(
        canonical_kpi=data.get("canonical_kpi"),
        value=data.get("value"),
        unit=data.get("unit"),
        period=data.get("period") or candidate.extracted_period,
        is_kpi=bool(data.get("is_kpi", False)),
        confidence=float(data.get("confidence", 0.0)),
        reason=data.get("reason", ""),
        source_type=candidate.source_type,
        page_num=candidate.page_num,
        section_hint=candidate.section_hint,
        section_type=candidate.section_type,
        section_score=candidate.section_score,
        label_text=candidate.label_text,
        value_text=candidate.value_text,
        normalized_value_text=candidate.normalized_value_text,
        extracted_period=candidate.extracted_period,
        raw_text=candidate.raw_text,
        normalization_source="llm"
    )


def normalize_candidate(candidate: Candidate) -> Optional[NormalizedKPI]:
    dict_result = normalize_candidate_with_dictionary(candidate)
    if dict_result is not None:
        return dict_result
    return llm_normalize_candidate(candidate)


def consolidate_kpis(items: List[NormalizedKPI]) -> List[dict]:
    filtered = [
        x for x in items
        if x is not None
        and x.is_kpi
        and x.canonical_kpi in CANONICAL_KPIS
        and x.confidence >= 0.60
        and x.value is not None
    ]

    best_by_key = {}
    for item in filtered:
        key = (item.canonical_kpi, item.period, item.unit)

        score = (
            item.confidence,
            SOURCE_PRIORITY.get(item.source_type, 0),
            item.section_score,
            1 if item.normalization_source == "dictionary" else 0,
            1 if item.section_hint else 0,
        )

        if key not in best_by_key or score > best_by_key[key][0]:
            best_by_key[key] = (score, item)

    return [asdict(v[1]) for v in best_by_key.values()]


## 9. Derived metrics

In [ ]:
def build_kpi_dict(normalized_kpis: List[dict]) -> Dict[str, Any]:
    result = {}
    for item in normalized_kpis:
        kpi = item.get("canonical_kpi")
        value = item.get("value")
        if kpi and value is not None and kpi not in result:
            result[kpi] = value
    return result


def safe_div(a, b):
    if a is None or b is None or b == 0:
        return None
    return a / b


def compute_derived_metrics(kpis: Dict[str, Any]) -> Dict[str, Any]:
    revenue = kpis.get("revenue")
    ebitda = kpis.get("ebitda")
    net_income = kpis.get("net_income")
    debt = kpis.get("debt")
    equity = kpis.get("equity")
    cash = kpis.get("cash")
    operating_income = kpis.get("operating_income")

    derived = {
        "ebitda_margin": safe_div(ebitda, revenue),
        "net_margin": safe_div(net_income, revenue),
        "operating_margin": safe_div(operating_income, revenue),
        "debt_to_equity": safe_div(debt, equity),
        "cash_to_debt": safe_div(cash, debt),
    }

    return {k: v for k, v in derived.items() if v is not None}


## 10. Analytical report and market decision

In [ ]:
def build_analytical_report_prompt(report_input: Dict[str, Any]) -> str:
    return f"""
You are a senior equity research analyst.

Your task is to prepare a concise analytical report based ONLY on the structured KPI data below.
Do not use outside knowledge.
Do not invent missing facts.
If the data is mixed or insufficient, say so explicitly.
Write the analytical report in Russian.

Company: {report_input.get("company")}
Ticker: {report_input.get("ticker")}
Event date: {report_input.get("event_date")}
Filing type: {report_input.get("filing_type")}
Fiscal period: {report_input.get("fiscal_period")}
Currency: {report_input.get("currency")}

Normalized KPIs:
{json.dumps(report_input.get("normalized_kpis", {}), ensure_ascii=False, indent=2)}

Derived metrics:
{json.dumps(report_input.get("derived_metrics", {}), ensure_ascii=False, indent=2)}

Write the output as valid JSON only with the following schema:
{{
  "executive_summary": "string",
  "positive_factors": ["string", "string"],
  "risk_factors": ["string", "string"],
  "financial_health_assessment": "string",
  "profitability_assessment": "string",
  "leverage_assessment": "string",
  "investment_view": {{
    "signal": "strong_buy | buy | hold | sell | strong_sell",
    "confidence": 0.0,
    "expected_short_term_reaction": "string",
    "rationale": "string"
  }},
  "key_kpi_interpretation": [
    {{
      "kpi": "string",
      "interpretation": "string"
    }}
  ]
}}
""".strip()


def generate_analytical_report(report_input: Dict[str, Any], max_new_tokens: int = 900) -> Dict[str, Any]:
    result = run_llm_json(build_analytical_report_prompt(report_input), max_new_tokens=max_new_tokens)
    return {
        "prompt": result["prompt"],
        "raw_response": result["raw_response"],
        "parsed_report": result["parsed"]
    }


def build_market_decision_prompt(report_input: Dict[str, Any]) -> str:
    return f"""
You are a senior equity research analyst.

Your task is to produce ONLY a compact market decision based ONLY on the KPI data below.
Do not use outside knowledge.
Do not invent missing facts.
If the data is mixed or insufficient, prefer a neutral signal.
Write the output in Russian.

Company: {report_input.get("company")}
Ticker: {report_input.get("ticker")}
Event date: {report_input.get("event_date")}
Filing type: {report_input.get("filing_type")}
Fiscal period: {report_input.get("fiscal_period")}
Currency: {report_input.get("currency")}

Normalized KPIs:
{json.dumps(report_input.get("normalized_kpis", {}), ensure_ascii=False, indent=2)}

Derived metrics:
{json.dumps(report_input.get("derived_metrics", {}), ensure_ascii=False, indent=2)}

Return valid JSON only with this schema:
{{
  "signal": "strong_buy | buy | hold | sell | strong_sell",
  "confidence": 0.0,
  "expected_move": "string",
  "rationale": "string"
}}
""".strip()


def generate_market_decision(report_input: Dict[str, Any], max_new_tokens: int = 350) -> Dict[str, Any]:
    result = run_llm_json(build_market_decision_prompt(report_input), max_new_tokens=max_new_tokens)
    return {
        "prompt": result["prompt"],
        "raw_response": result["raw_response"],
        "parsed_decision": result["parsed"]
    }


def render_report_markdown(report: Dict[str, Any]) -> str:
    if not report:
        return "Не удалось распарсить JSON-ответ модели."

    inv = report.get("investment_view", {})
    kpi_items = report.get("key_kpi_interpretation", [])

    lines = []
    lines.append("# Аналитический отчет\n")
    lines.append(f"## Executive Summary\n{report.get('executive_summary', '')}\n")

    lines.append("## Positive Factors")
    for x in report.get("positive_factors", []):
        lines.append(f"- {x}")
    lines.append("")

    lines.append("## Risk Factors")
    for x in report.get("risk_factors", []):
        lines.append(f"- {x}")
    lines.append("")

    lines.append(f"## Financial Health Assessment\n{report.get('financial_health_assessment', '')}\n")
    lines.append(f"## Profitability Assessment\n{report.get('profitability_assessment', '')}\n")
    lines.append(f"## Leverage Assessment\n{report.get('leverage_assessment', '')}\n")

    lines.append("## Investment View")
    lines.append(f"- Signal: **{inv.get('signal', '')}**")
    lines.append(f"- Confidence: **{inv.get('confidence', '')}**")
    lines.append(f"- Expected short-term reaction: **{inv.get('expected_short_term_reaction', '')}**")
    lines.append(f"- Rationale: {inv.get('rationale', '')}")
    lines.append("")

    lines.append("## KPI Interpretation")
    for item in kpi_items:
        lines.append(f"- **{item.get('kpi', '')}**: {item.get('interpretation', '')}")

    return "\n".join(lines)


## 11. Основная end-to-end функция

In [ ]:
def run_end_to_end_pipeline(
    pdf_path: str,
    company: str,
    ticker: str,
    event_date: str,
    filing_type: str,
    fiscal_period: str,
    currency: str = "RUB"
) -> Dict[str, Any]:
    pages = extract_pages(pdf_path)
    candidates = generate_candidates(pages)

    print(f"Pages: {len(pages)}")
    print(f"Candidates: {len(candidates)}")

    normalized_items = []
    failed_candidates = []
    llm_calls = 0
    dictionary_hits = 0

    for i, cand in enumerate(candidates, start=1):
        try:
            item = normalize_candidate(cand)
            if item is not None:
                if item.normalization_source == "dictionary":
                    dictionary_hits += 1
                else:
                    llm_calls += 1
            normalized_items.append(item)
        except Exception as e:
            failed_candidates.append({
                "candidate": asdict(cand),
                "error": str(e)
            })
            print(f"[WARN] candidate {i} failed: {e}")

    normalized_kpis_list = consolidate_kpis(normalized_items)
    normalized_kpis_dict = build_kpi_dict(normalized_kpis_list)
    derived_metrics = compute_derived_metrics(normalized_kpis_dict)

    report_input = {
        "company": company,
        "ticker": ticker,
        "event_date": event_date,
        "filing_type": filing_type,
        "fiscal_period": fiscal_period,
        "currency": currency,
        "normalized_kpis": normalized_kpis_dict,
        "derived_metrics": derived_metrics,
    }

    analytical_result = generate_analytical_report(report_input)
    decision_result = generate_market_decision(report_input)

    return {
        "metadata": {
            "pdf_path": pdf_path,
            "company": company,
            "ticker": ticker,
            "event_date": event_date,
            "filing_type": filing_type,
            "fiscal_period": fiscal_period,
            "currency": currency,
            "num_pages": len(pages),
            "num_candidates": len(candidates),
            "dictionary_hits": dictionary_hits,
            "llm_calls_for_normalization": llm_calls,
        },
        "debug_pages": pages,
        "debug_candidates_raw": [asdict(c) for c in candidates],
        "debug_candidates_normalized_before_consolidation": [
            asdict(x) for x in normalized_items if x is not None
        ],
        "debug_failed_candidates": failed_candidates,
        "normalized_kpis_list": normalized_kpis_list,
        "normalized_kpis_dict": normalized_kpis_dict,
        "derived_metrics": derived_metrics,
        "analytical_report": analytical_result["parsed_report"],
        "market_decision": decision_result["parsed_decision"],
        "raw_analytical_response": analytical_result["raw_response"],
        "raw_decision_response": decision_result["raw_response"],
    }


## 12. Запуск пайплайна

In [ ]:
company = "Название компании"
ticker = "TICKER"
event_date = "2024-12-31"
filing_type = "annual_report"
fiscal_period = "FY2024"
currency = "RUB"

pipeline_result = run_end_to_end_pipeline(
    pdf_path=pdf_path,
    company=company,
    ticker=ticker,
    event_date=event_date,
    filing_type=filing_type,
    fiscal_period=fiscal_period,
    currency=currency
)

pipeline_result["normalized_kpis_dict"]


## 13. Derived metrics

In [ ]:
pipeline_result["derived_metrics"]

## 14. Market decision

In [ ]:
pipeline_result["market_decision"]

## 15. Human-readable analytical report

In [ ]:
print(render_report_markdown(pipeline_result["analytical_report"]))

## 16. Debug: сырые кандидаты

In [ ]:
pipeline_result["debug_candidates_raw"][:10]

## 17. Debug: кандидаты после нормализации

In [ ]:
pipeline_result["debug_candidates_normalized_before_consolidation"][:10]

## 18. Debug: pages / section hints

In [ ]:
pipeline_result["debug_pages"][:3]

## 19. Сохранение результата

In [ ]:
from google.colab import files

with open("end_to_end_pipeline_result_v3.json", "w", encoding="utf-8") as f:
    json.dump(pipeline_result, f, ensure_ascii=False, indent=2)

print("Saved to end_to_end_pipeline_result_v3.json")
files.download("end_to_end_pipeline_result_v3.json")


## Что изменилось в v3

1. Усилен `section_hint`:
   - section type
   - section score
   - header-zone scoring
2. Добавлен словарный слой до LLM.
3. Очевидные KPI нормализуются без LLM.
4. Сохраняется статистика:
   - `dictionary_hits`
   - `llm_calls_for_normalization`
5. Для отладки сохраняются page-level section signals.

Эта версия быстрее и дешевле по числу LLM-вызовов, чем v2.
